# 3. Stored results discovery and analysis

This notebook reads metadata and saved arrays. It never performs propagation or Fourier fitting.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the checkout so notebooks work without an editable install.
# Supported working directories: repository root or a directory inside it.
_here = Path.cwd().resolve()
ROOT = next(
    (directory for directory in (_here, *_here.parents)
     if (directory / "src" / "bosonic_algo" / "__init__.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the bosonic_algo checkout from the notebook kernel's "
        "working directory. Start Jupyter in the repository root or its "
        "notebooks/ folder (or change the kernel working directory). "
        "For another environment, install from the repository root with "
        "'python -m pip install -e .'."
    )
_src = str(ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from bosonic_algo import ResultStore
store = ResultStore(ROOT / "results")
metadata = store.list_results()
pd.DataFrame([{"run_id":m["run_id"], "method":m["method"], "fit_id":m["inputs"].get("fit_id"), "potential":m["inputs"].get("potential"), "cavity":m["inputs"]["config"]["cavity"]["enabled"], "created":m["created_utc"]} for m in metadata])

## Search a subset

Use `fit_id=...`, `include_cavity=...`, or `**{"config.tgifs.substeps": 20}` to filter metadata.

In [ ]:
matches = store.find_results(method="tgifs", include_cavity=False)
[(m["run_id"], m["inputs"].get("fit_id")) for m in matches]

## Retrieve and plot any saved run

Set an existing run ID and method from the table above; no simulation is performed.

In [ ]:
METHOD = "tgifs"
RUN_ID = None
if RUN_ID is None:
    print("Set RUN_ID to load a saved result")
else:
    data, meta = store.load(METHOD, RUN_ID)
    print(meta["inputs"])
    import matplotlib.pyplot as plt
    plt.plot(data["times"], data["mean_R"], label="<R>")
    plt.legend()